**Early experiment. The project itself is `notebook/model.ipynb`.**

# Experiment: does asking NLLB for more options fix gender and formality?

Our picker chooses, from NLLB's list of Serbian translations, the first one whose endings match what we know about the conversation (who speaks, who listens, how formal). The model check showed a problem: sometimes **none** of NLLB's top 8 options has the right ending (for 你昨天来上班了吗？ said to a woman, all 8 said došao, none said došla). Then the picker has nothing to pick.

**Fix #1, tested here:** ask NLLB for more options (16 or 32 instead of 8), or for more *varied* options, and measure how often at least one of them has the right endings.

What this notebook does:
1. Loads our 60 draft Mandarin test sentences with their labels.
2. Defines rough detectors that read a Serbian sentence (in Latin script) and say: speaker woman or man? listener woman or man? formal or informal?
3. Asks NLLB for options four ways: beam search with 8, 16 and 32 options, and 16 random samples.
4. Converts every option from Cyrillic to Latin script, since our project writes Serbian in Latin script.
5. Counts, for each way, how often some option fits the labels, how often the top option already fits, and how far down the list the first fitting option is.

**Read the results with care:**
- The detectors are a **rough first version written by code, not by a Serbian speaker**. Our Serbian speaker's rules replaced them later (in `model.ipynb`). They can miss correct forms or accept wrong ones.
- The sentences are **AI drafts**. When this ran, our Mandarin speaker hadn't checked the Mandarin yet and there was no human-written Serbian.
- "Fits the labels" only means the *endings* look right. An option can fit and still mean the wrong thing or contain a non-Serbian word (section 9 hunts for some of those).
- Our fair-test rule says nobody studies test-row output before the rules are final. So the tables below give **counts** for all rows, but the sentence-by-sentence examples come from the **practice rows only**.

**How to run:** Runtime → Run all. A GPU is optional. The model download takes a few minutes the first time.

## 1. Setup

Install two small libraries the translation model needs, then fix the random seed so every run gives the same results.

In [1]:
# Colab already has transformers, torch and pandas. These two split text into word pieces for the model.
%pip install -q sentencepiece sacremoses

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re, time, random
from collections import Counter
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Use the GPU if Colab gave us one; otherwise the CPU (slower, still fine for 60 sentences).
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)
pd.set_option("display.max_colwidth", None)

Running on: cpu


## 2. Load the 60 draft test sentences

The file lives in our GitHub repo. We load it from the repo's public web address. If that address can't be reached, the code falls back to the copy in the repo folder (`data/draft_mandarin_60.csv`). The saved run below was made while the repo was still private, so it used the local copy.

Each row has the Mandarin sentence and the labels: `speaker_gender`, `listener_gender`, `formal` (yes/no), `what_it_tests` (which endings the sentence checks) and `split` (practice or test).

In [3]:
URL = "https://raw.githubusercontent.com/proelium27/veritas-ai-hackathon/main/data/draft_mandarin_60.csv"
LOCAL_COPIES = ["data/draft_mandarin_60.csv", "../../data/draft_mandarin_60.csv"]

try:
    sentences = pd.read_csv(URL)
    print("Loaded from the repo's public address.")
except Exception as error:
    print(f"Couldn't load from the web address ({type(error).__name__}); using the local copy.")
    for path in LOCAL_COPIES:
        try:
            sentences = pd.read_csv(path)
            print("Loaded", path)
            break
        except FileNotFoundError:
            pass
    else:
        raise FileNotFoundError("No copy of draft_mandarin_60.csv found. Make the repo public or upload the file.")

print(len(sentences), "sentences")
print(sentences["what_it_tests"].value_counts().to_string())
sentences[["id", "mandarin", "speaker_gender", "listener_gender", "formal", "what_it_tests", "split"]].head(8)

Couldn't load from the web address (HTTPError); using the local copy.
Loaded data/draft_mandarin_60.csv
60 sentences
what_it_tests
speaker_gender               18
formality                    12
listener_gender+formality    12
listener_gender               6
speaker_gender+formality      6
control                       6


,id,mandarin,speaker_gender,listener_gender,formal,what_it_tests,split
0,t01,我昨天加了两个小时的班。,woman,man,yes,speaker_gender,practice
1,t02,我把报告写完了。,man,woman,yes,speaker_gender,practice
2,t03,我已经到车间了。,woman,woman,no,speaker_gender,practice
3,t04,我今天太累了。,man,man,no,speaker_gender,test
4,t05,我忘了带安全帽。,woman,group,no,speaker_gender,test
5,t06,我刚才看见叉车撞到货架了。,man,woman,yes,speaker_gender,practice
6,t07,我已经换好工作服了。,woman,man,no,speaker_gender,test
7,t08,我昨天生病了。,man,man,yes,speaker_gender,test


## 3. Load NLLB and the four ways of asking for options

`facebook/nllb-200-distilled-600M` translates directly from Mandarin (`zho_Hans`) to Serbian. NLLB only offers Serbian in Cyrillic (`srp_Cyrl`), but our project uses Serbian in **Latin script**, so we convert every option with `to_latin()` (next cell). Serbian's two alphabets match letter for letter, so nothing is lost.

The four ways:
- **beam 8 / beam 16 / beam 32**: beam search keeps the N most likely partial translations at each step and returns all N at the end, most likely first. Beam 8 was what the picker used when we ran this; after this experiment it uses 32.
- **sample 16**: instead of always keeping the most likely words, the model *draws* each next word at random in proportion to how likely it is (only among the words that make up the top 90% of the probability, called top-p = 0.9). We draw 16 translations. This gives more varied options than beam search, which tends to return 16 near-copies of the same sentence.

**Why sampling and not "diverse beam search"?** The plan was diverse beam search (beam groups pushed apart by a diversity penalty). In current versions of the transformers library that method was moved out of the library into code downloaded from the Hugging Face Hub, which only runs with `trust_remote_code=True`, meaning it runs code we haven't read. We chose not to do that, so sampling is our "more varied" method. Its random draws use a fixed seed per sentence, so a rerun gives the same options on the same machine (a GPU can give slightly different draws than a CPU).

We remove exact duplicates from every list. For sampling we sort the options by how many times they were drawn (most often first), so "top option" means the one the model produced most often.

In [4]:
# Serbian Cyrillic -> Serbian Latin, letter for letter. Serbian has an exact one-to-one match
# between the two alphabets (three Cyrillic letters become two Latin letters: Љ→Lj, Њ→Nj, Џ→Dž).
CYR_TO_LAT = {
    "а": "a", "б": "b", "в": "v", "г": "g", "д": "d", "ђ": "đ", "е": "e", "ж": "ž", "з": "z",
    "и": "i", "ј": "j", "к": "k", "л": "l", "љ": "lj", "м": "m", "н": "n", "њ": "nj", "о": "o",
    "п": "p", "р": "r", "с": "s", "т": "t", "ћ": "ć", "у": "u", "ф": "f", "х": "h", "ц": "c",
    "ч": "č", "џ": "dž", "ш": "š",
}

def to_latin(text):
    out = []
    for i, ch in enumerate(text):
        low = ch.lower()
        if low not in CYR_TO_LAT:
            out.append(ch)          # punctuation, digits, and non-Serbian letters (я, ы, й) stay as they are
            continue
        lat = CYR_TO_LAT[low]
        if ch.isupper():
            # Capital Љ/Њ/Џ: "Lj" at the start of a normal word, "LJ" inside an ALL-CAPS word
            next_is_upper = i + 1 < len(text) and text[i + 1].isupper()
            lat = lat.upper() if next_is_upper else lat.capitalize()
        out.append(lat)
    return "".join(out)

print(to_latin("Проверила сам. Љубав, Њујорк, Џеп, Ђорђе, Ћуприја"))

Proverila sam. Ljubav, Njujork, Džep, Đorđe, Ćuprija


In [5]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
start = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="zho_Hans")
nllb = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE).eval()
SERBIAN = tokenizer.convert_tokens_to_ids("srp_Cyrl")   # forces the output to be Serbian Cyrillic
print(f"Loaded {MODEL_NAME} in {time.time() - start:.0f} s")

METHODS = {
    "beam 8":    dict(num_beams=8,  num_return_sequences=8),
    "beam 16":   dict(num_beams=16, num_return_sequences=16),
    "beam 32":   dict(num_beams=32, num_return_sequences=32),
    "sample 16": dict(do_sample=True, top_p=0.9, num_beams=1, num_return_sequences=16),
}

def get_options(mandarin, method, sentence_number):
    """Translate one Mandarin sentence with one method.
    Returns (list of unique options in Cyrillic, seconds taken). We convert to Latin after saving."""
    torch.manual_seed(SEED + sentence_number)       # same random draws every run, different per sentence
    batch = tokenizer([mandarin], return_tensors="pt").to(DEVICE)
    start = time.time()
    with torch.no_grad():
        out = nllb.generate(**batch, forced_bos_token_id=SERBIAN, max_new_tokens=64, **METHODS[method])
    seconds = time.time() - start
    options = [o.strip() for o in tokenizer.batch_decode(out, skip_special_tokens=True)]
    if method.startswith("sample"):
        options = [o for o, count in Counter(options).most_common()]   # most often drawn first
    else:
        options = list(dict.fromkeys(options))                         # drop repeats, keep beam order
    return options, seconds

cyrillic_options, seconds = get_options("我检查过了。", "beam 8", 0)
print([to_latin(o) for o in cyrillic_options])

Loaded facebook/nllb-200-distilled-600M in 1 s


['Proverio sam.', 'Pogledao sam.', 'Proveril sam.', 'Proverila sam.', 'Ja sam proverila.', 'Proveriv sam.', 'Proverena sam.']


## 4. Rough detectors: what does a Serbian sentence show? (first version, later replaced)

Serbian shows gender and politeness in word endings:

| Question | Woman / formal | Man / informal | Where we look |
|---|---|---|---|
| Speaker ("I did") | Proveri**la** sam | Proveri**o** sam | the word just before **sam** (I am) / **nisam** / **jesam**, or up to 3 words after it (Ja sam **ga** isključi**la**) |
| Speaker ("I am tired") | umorna, spremna, bolesna | umoran, spreman, bolestan | same place |
| Listener ("did you…", informal) | Da li si doš**la** | Da li si doš**ao** | around **si** / **nisi** / **jesi** |
| Formality | Vi, Vas, Vam, Vaš…, ste, Jeste, verbs ending -ete/-ite/-ate (možete, potpišite) | ti, tebi, si, Jesi, tvoj…, verbs ending -eš/-iš/-aš (možeš, znaš) | anywhere in the sentence |

Each detector returns `None` when the sentence doesn't show the answer, and `"mixed"` when it finds both kinds (for example a feminine and a masculine verb).

Known weak spots of this first version:
- `te` (informal "you", as in "video sam te") is left out because it also means "those" (te kutije) and "and so".
- Informal commands (Idi, Potpiši) aren't detected: too many Serbian words end in -i.
- The gender patterns only look next to sam/si, so "Ja sam juče na poslu u fabrici radila" (the verb more than 3 words away) is missed.
- Any word ending in -la next to sam counts as feminine, so a noun like "škola" right before sam would fool it.

In [6]:
# ROUGH FIRST VERSION of the Serbian ending detectors, for Serbian in LATIN script.
# Written by code, not by a Serbian speaker. Our Serbian speaker's real rules replaced these later.
# Each detector reads ONE Serbian sentence and answers a question about it,
# or returns None when the sentence doesn't show the answer.
import re

# Split a sentence into lowercase words (punctuation dropped).
def words(sentence):
    return re.findall(r"\w+", sentence.lower())

# "I am / I am not / am I": the words that carry first person
I_AM = {"sam", "nisam", "jesam"}
# "you are / you are not / are you" for ONE person, informal (ti form)
YOU_ARE = {"si", "nisi", "jesi"}

# Past-tense verbs ("I checked", "you came") end differently for a woman and a man:
#   woman: proveri-LA, doš-LA, bi-LA        man: proveri-O, doš-AO, bi-O, je-O, ču-O
FEMININE_VERB = re.compile(r"\w+la$")
MASCULINE_VERB = re.compile(r"\w*(ao|io|eo|uo)$")
LOOKS_LIKE_A_VERB_BUT_ISNT = {"kao"}   # "kao" = "like / as"

# A few describing words (adjectives) whose ending shows gender: man -> woman
ADJECTIVES = {
    "spreman": "spremna",     # ready
    "umoran": "umorna",       # tired
    "bolestan": "bolesna",    # sick
    "gotov": "gotova",        # done / ready
    "povređen": "povređena",  # hurt
    "siguran": "sigurna",     # sure
    "gladan": "gladna",       # hungry
    "zauzet": "zauzeta",      # busy
}
MASCULINE_ADJ = set(ADJECTIVES)
FEMININE_ADJ = set(ADJECTIVES.values())

def word_gender(w):
    """'woman', 'man' or None for one word."""
    if w in LOOKS_LIKE_A_VERB_BUT_ISNT:
        return None
    if w in FEMININE_ADJ or FEMININE_VERB.match(w):
        return "woman"
    if w in MASCULINE_ADJ or MASCULINE_VERB.match(w):
        return "man"
    return None

def gender_near(sentence, anchor_words, words_after=3):
    """Look at the word just BEFORE each anchor word ("Proverila sam")
    and up to `words_after` words AFTER it ("Ja sam ga isključila", "Da li si juče došla").
    Returns 'woman', 'man', 'mixed' (both found) or None (neither found)."""
    found = set()
    # Split at punctuation first so we never look across a comma or full stop.
    for chunk in re.split(r"[.,!?;:—–-]", sentence):
        ws = words(chunk)
        for i, w in enumerate(ws):
            if w not in anchor_words:
                continue
            if i > 0 and word_gender(ws[i - 1]):
                found.add(word_gender(ws[i - 1]))
            for nxt in ws[i + 1:i + 1 + words_after]:
                g = word_gender(nxt)
                if g:
                    found.add(g)
                    break   # stop at the first gendered word after the anchor
    if len(found) == 2:
        return "mixed"
    return found.pop() if found else None

def detect_speaker_gender(sentence):
    """Is the speaker ('I') a woman or a man? Looks around sam / nisam / jesam."""
    return gender_near(sentence, I_AM)

def detect_listener_gender(sentence):
    """Is the listener ('you', informal, one person) a woman or a man? Looks around si / nisi / jesi.
    Formal Vi takes the plural (došli) for everyone, so formal sentences give None here."""
    return gender_near(sentence, YOU_ARE)

# Formal "you" (Vi form) vs informal "you" (ti form)
FORMAL_WORDS = {"vi", "vas", "vam", "vama", "ste", "jeste", "niste", "biste"}
INFORMAL_WORDS = {"ti", "tebe", "tebi", "tobom", "si", "jesi", "nisi"}
# "te" (you, object) is left out on purpose: it also means "those" (te kutije) and "and so",
# which would cause false alarms. A smarter rule for it came later, in model.ipynb.
NOT_VERBS = {"naš", "dete", "plate", "pakete"}  # words that only LOOK like 'you' verb endings

def detect_formality(sentence):
    """'formal', 'informal', 'mixed' (both kinds found) or None (no 'you' at all)."""
    formal = informal = False
    for w in words(sentence):
        if w in FORMAL_WORDS or w.startswith("vaš"):              # Vi, Vas, Vaš paket
            formal = True
        elif w in INFORMAL_WORDS or re.match(r"tvo(j|g|m)", w):   # ti, si, tvoj
            informal = True
        elif w in NOT_VERBS:
            continue
        # Verb endings: možete / potpišite (formal, plural) vs možeš / znaš (informal, singular)
        elif len(w) >= 5 and re.search(r"(ete|ite|ate)$", w):
            formal = True
        elif len(w) >= 4 and re.search(r"(eš|iš|aš)$", w):
            informal = True
    if formal and informal:
        return "mixed"
    if formal:
        return "formal"
    if informal:
        return "informal"
    return None

### Test the detectors on hand-written sentences

Before trusting the detectors on NLLB's output, we check them on short sentences whose answers we know. Each line shows the sentence, what we expect (speaker, listener, formality) and what the detectors said.

In [7]:
known_answers = [
    # sentence                                 speaker   listener  formality
    ("Proverila sam.",                        "woman",  None,     None),
    ("Proverio sam.",                         "man",    None,     None),
    ("Ja sam ga isključila.",                 "woman",  None,     None),
    ("Ja sam već završio izveštaj.",          "man",    None,     None),
    ("Umorna sam.",                           "woman",  None,     None),
    ("Bio sam bolestan.",                     "man",    None,     None),
    ("Nisam dobro čula.",                     "woman",  None,     None),
    ("Da li si juče došla na posao?",         None,     "woman",  "informal"),
    ("Jesi li umoran?",                       None,     "man",    "informal"),
    ("Jeste li došli na posao?",              None,     None,     "formal"),
    ("Možete li sutra doći ranije?",          None,     None,     "formal"),
    ("Možeš li sutra doći ranije?",           None,     None,     "informal"),
    ("Vaš paket je kod portira.",             None,     None,     "formal"),
    ("Tvoja kartica je ovde.",                None,     None,     "informal"),
    ("Potpišite ovde, molim Vas.",            None,     None,     "formal"),
    ("Poslala sam Vam to.",                   "woman",  None,     "formal"),
    ("Rekao sam ti već.",                     "man",    None,     "informal"),
    ("Mašina na liniji tri je opet stala.",   None,     None,     None),
    ("Proveril sam.",                         None,     None,     None),   # Russian form: not Serbian, so no answer
]

failures = 0
for sentence, *expected in known_answers:
    got = [detect_speaker_gender(sentence), detect_listener_gender(sentence), detect_formality(sentence)]
    ok = got == expected
    failures += not ok
    print("pass" if ok else "FAIL", f"{sentence:38s} expected {expected}, got {got}")
print(f"\n{len(known_answers) - failures} of {len(known_answers)} passed")
assert failures == 0, "A detector is wrong on a hand-written example"

pass Proverila sam.                         expected ['woman', None, None], got ['woman', None, None]
pass Proverio sam.                          expected ['man', None, None], got ['man', None, None]
pass Ja sam ga isključila.                  expected ['woman', None, None], got ['woman', None, None]
pass Ja sam već završio izveštaj.           expected ['man', None, None], got ['man', None, None]
pass Umorna sam.                            expected ['woman', None, None], got ['woman', None, None]
pass Bio sam bolestan.                      expected ['man', None, None], got ['man', None, None]
pass Nisam dobro čula.                      expected ['woman', None, None], got ['woman', None, None]
pass Da li si juče došla na posao?          expected [None, 'woman', 'informal'], got [None, 'woman', 'informal']
pass Jesi li umoran?                        expected [None, 'man', 'informal'], got [None, 'man', 'informal']
pass Jeste li došli na posao?               expected [None, None, 'formal'

## 5. What must each sentence show?

The `what_it_tests` label says which endings a sentence checks. From it we decide which properties a translation must show:

| `what_it_tests` | Must show |
|---|---|
| speaker_gender | the speaker's gender |
| listener_gender | the listener's gender (only for informal rows; formal Ви uses the plural for everyone) |
| formality | formal or informal, matching `formal` |
| speaker_gender+formality | both |
| listener_gender+formality | formality, plus the listener's gender when informal |
| control | nothing (these rows are left out of the scores) |

An option **fits** when every required property is detected *and* matches the label. If a detector returns `None` (not shown) or `"mixed"`, the option doesn't fit.

In [8]:
def required_properties(row):
    tests = row["what_it_tests"].split("+")
    need = {}
    if "speaker_gender" in tests:
        need["speaker"] = row["speaker_gender"]
    if "listener_gender" in tests and row["formal"] == "no":
        need["listener"] = row["listener_gender"]
    if "formality" in tests:
        need["formality"] = "formal" if row["formal"] == "yes" else "informal"
    return need

DETECTORS = {"speaker": detect_speaker_gender, "listener": detect_listener_gender, "formality": detect_formality}

def fits(option, need):
    return all(DETECTORS[prop](option) == wanted for prop, wanted in need.items())

sentences["need"] = sentences.apply(required_properties, axis=1)
sentences[["id", "mandarin", "what_it_tests", "formal", "need"]].iloc[[0, 18, 24, 36, 44, 54]]

,id,mandarin,what_it_tests,formal,need
0,t01,我昨天加了两个小时的班。,speaker_gender,yes,{'speaker': 'woman'}
18,t19,你吃午饭了吗？,listener_gender,no,{'listener': 'woman'}
24,t25,您需要帮忙吗？,formality,yes,{'formality': 'formal'}
36,t37,我已经给您发过去了。,speaker_gender+formality,yes,"{'speaker': 'woman', 'formality': 'formal'}"
44,t45,您签收那批货了吗？,listener_gender+formality,yes,{'formality': 'formal'}
54,t55,三号线的机器又停了。,control,no,{}


## 6. Generate the options (saved, so nothing is generated twice)

For every sentence and every method we ask NLLB for its options one sentence at a time, and time each call. NLLB's Cyrillic output is saved to `more_options_cache.csv` in Colab's working folder. If that file already exists, we load it instead of generating again. Delete it to start over. After loading, every option is converted to Latin script.

In [9]:
CACHE_FILE = "more_options_cache.csv"

try:
    options_table = pd.read_csv(CACHE_FILE)
    options_table["option_cyrillic"] = options_table["option_cyrillic"].fillna("")   # an empty translation reads back as blank
    print(f"Loaded {len(options_table)} saved options from {CACHE_FILE} (timings are from when they were generated).")
except FileNotFoundError:
    rows = []
    total_start = time.time()
    for number, row in sentences.iterrows():
        for method in METHODS:
            options, seconds = get_options(row["mandarin"], method, number)
            for rank, option in enumerate(options, 1):
                rows.append({"id": row["id"], "method": method, "rank": rank, "option_cyrillic": option, "seconds": seconds})
    options_table = pd.DataFrame(rows)
    options_table.to_csv(CACHE_FILE, index=False)
    print(f"Generated {len(options_table)} options in {time.time() - total_start:.0f} s, saved to {CACHE_FILE}")

# Our project writes Serbian in Latin script: convert every option. From here on we only use the Latin column.
options_table["option"] = options_table["option_cyrillic"].map(to_latin)
print(options_table.groupby("method", sort=False)["id"].count().rename("options (all sentences)").to_string())
options_table[["id", "method", "rank", "option_cyrillic", "option"]].head(5)

Generated 4183 options in 273 s, saved to more_options_cache.csv
method
beam 8        465
beam 16       930
beam 32      1847
sample 16     941


,id,method,rank,option_cyrillic,option
0,t01,beam 8,1,У суботу сам додао два сата.,U subotu sam dodao dva sata.
1,t01,beam 8,2,У суботу сам додао два сата рада.,U subotu sam dodao dva sata rada.
2,t01,beam 8,3,У јуче сам додао два сата рада.,U juče sam dodao dva sata rada.
3,t01,beam 8,4,Учера сам додао два сата рада.,Učera sam dodao dva sata rada.
4,t01,beam 8,5,Учера сам додао два сата.,Učera sam dodao dva sata.


## 7. Score every sentence and method

For each sentence and method we record:
- **some option fits**: at least one option has all the required endings (the best a picker could do),
- **top option fits**: the first option already fits (for beam search, the top option is what NLLB gives by itself),
- **rank of first fit**: how far down the list the first fitting option is,
- **clean fit**: some fitting option has no wrong-word warning from section 9's checker.

In [10]:
# Hints that an option may contain a WRONG word (not a gender question). Only hints, for our Serbian speaker to check.
SERBIAN_LATIN_LETTERS = set("abcčćdđefghijklmnoprsštuvzž")
NOT_SERBIAN_WORDS = {"včera": "Russian/Bulgarian for 'yesterday' (Serbian: juče)",
                     "učora": "Ukrainian for 'yesterday' (Serbian: juče)",
                     "učera": "not a word; a mix of juče and včera (Serbian: juče)",
                     "uključio": "means 'turned on', check the meaning",
                     "uključila": "means 'turned on', check the meaning"}

def wrong_word_hints(sentence):
    """A list of reasons this sentence might contain a non-Serbian or wrong word (empty list = none seen)."""
    hints = []
    for w in words(sentence):
        if w in NOT_SERBIAN_WORDS:
            hints.append(f"{w}: {NOT_SERBIAN_WORDS[w]}")
        elif any(ch.isalpha() and ch not in SERBIAN_LATIN_LETTERS for ch in w):
            # to_latin() leaves Russian/Ukrainian letters (я, ы, й ...) as they are, so they show up here
            hints.append(f"{w}: has a letter Serbian doesn't use")
    # Russian-style past tense next to sam/si: "Proveril sam" (Serbian: Proverio / Proverila)
    for chunk in re.split(r"[.,!?;:]", sentence.lower()):
        for m in re.finditer(r"\b(\w+(?:il|el|al)) (?:sam|si)\b|\b(?:sam|si) (\w+(?:il|el|al))\b", chunk):
            hints.append(f"{m.group(1) or m.group(2)}: looks like a Russian past tense (Serbian ends in -o / -la)")
    return hints

In [11]:
METHOD_ORDER = list(METHODS)
scored = []
for _, row in sentences.iterrows():
    if not row["need"]:          # control rows: nothing to check
        continue
    for method in METHOD_ORDER:
        opts = options_table[(options_table["id"] == row["id"]) & (options_table["method"] == method)]
        opts = opts.sort_values("rank")["option"].tolist()
        fit_ranks = [i for i, o in enumerate(opts, 1) if fits(o, row["need"])]
        scored.append({
            "id": row["id"], "method": method, "what_it_tests": row["what_it_tests"], "split": row["split"],
            "speaker": row["speaker_gender"] if "speaker" in row["need"] else None,
            "listener": row["listener_gender"] if "listener" in row["need"] else None,
            "n_options": len(opts),
            "some_fit": bool(fit_ranks),
            "top_fits": 1 in fit_ranks,
            "first_fit_rank": fit_ranks[0] if fit_ranks else np.nan,
            "clean_fit": any(not wrong_word_hints(opts[r - 1]) for r in fit_ranks),
            "seconds": options_table[(options_table["id"] == row["id"]) & (options_table["method"] == method)]["seconds"].iloc[0],
        })
scored = pd.DataFrame(scored)
print(scored["id"].nunique(), "sentences scored (controls left out)")

54 sentences scored (controls left out)


## 8. Results

### 8a. The main table

Percentages are over the 54 non-control sentences. "Median rank" is over the sentences where some option fits; because the top option already fits in about half of them, we also give the median rank over the sentences where the top option misses. Seconds are per sentence on this run's hardware (see the setup cell for CPU or GPU).

In [12]:
def pct(flags):
    return f"{flags.mean():.0%} ({int(flags.sum())}/{len(flags)})"

def summary(df):
    return pd.Series({
        "some option fits": pct(df["some_fit"]),
        "top option fits": pct(df["top_fits"]),
        "fit with no wrong-word warning": pct(df["clean_fit"]),
        "median rank of first fit": df["first_fit_rank"].median(),
        "median rank when the top option misses": df.loc[~df["top_fits"], "first_fit_rank"].median(),
        "unique options (avg)": round(df["n_options"].mean(), 1),
        "seconds per sentence": round(df["seconds"].mean(), 2),
    })

main_table = pd.DataFrame({m: summary(scored[scored["method"] == m]) for m in METHOD_ORDER}).T
main_table

,some option fits,top option fits,fit with no wrong-word warning,median rank of first fit,median rank when the top option misses,unique options (avg),seconds per sentence
beam 8,80% (43/54),50% (27/54),80% (43/54),1.0,3.5,7.8,0.62
beam 16,83% (45/54),48% (26/54),83% (45/54),1.0,3.0,15.5,0.71
beam 32,91% (49/54),48% (26/54),91% (49/54),1.0,5.0,30.8,0.88
sample 16,85% (46/54),30% (16/54),85% (46/54),2.0,4.5,15.6,2.21


### 8b. Some option fits, by what the sentence tests

In [13]:
def fit_table(df, by):
    return df.pivot_table(index=by, columns="method", values="some_fit", aggfunc=pct, sort=False)[METHOD_ORDER]

fit_table(scored, "what_it_tests")

method,beam 8,beam 16,beam 32,sample 16
what_it_tests,,,,
speaker_gender,72% (13/18),78% (14/18),89% (16/18),78% (14/18)
listener_gender,67% (4/6),83% (5/6),83% (5/6),83% (5/6)
formality,92% (11/12),83% (10/12),92% (11/12),100% (12/12)
speaker_gender+formality,67% (4/6),83% (5/6),100% (6/6),83% (5/6)
listener_gender+formality,92% (11/12),92% (11/12),92% (11/12),83% (10/12)


In [14]:
# The same breakdown for "top option fits" (what NLLB gets right with no picker at all)
scored.pivot_table(index="what_it_tests", columns="method", values="top_fits", aggfunc=pct, sort=False)[METHOD_ORDER]

method,beam 8,beam 16,beam 32,sample 16
what_it_tests,,,,
speaker_gender,44% (8/18),44% (8/18),44% (8/18),28% (5/18)
listener_gender,17% (1/6),17% (1/6),17% (1/6),17% (1/6)
formality,75% (9/12),67% (8/12),67% (8/12),42% (5/12)
speaker_gender+formality,50% (3/6),50% (3/6),50% (3/6),0% (0/6)
listener_gender+formality,50% (6/12),50% (6/12),50% (6/12),42% (5/12)


### 8c. By speaker gender and listener gender

Only sentences where that gender must show are counted.

In [15]:
print("Speaker gender (sentences that must show it):")
display(fit_table(scored[scored["speaker"].notna()], "speaker"))
print("Listener gender (informal sentences that must show it):")
display(fit_table(scored[scored["listener"].notna()], "listener"))

Speaker gender (sentences that must show it):


method,beam 8,beam 16,beam 32,sample 16
speaker,,,,
woman,50% (6/12),67% (8/12),83% (10/12),58% (7/12)
man,92% (11/12),92% (11/12),100% (12/12),100% (12/12)


Listener gender (informal sentences that must show it):


method,beam 8,beam 16,beam 32,sample 16
listener,,,,
woman,67% (4/6),83% (5/6),83% (5/6),50% (3/6)
man,83% (5/6),83% (5/6),83% (5/6),100% (6/6)


In [16]:
# Top option fits, split by gender: does NLLB on its own favour men?
print("Top option fits, by speaker:")
display(scored[scored["speaker"].notna()].pivot_table(index="speaker", columns="method", values="top_fits", aggfunc=pct)[METHOD_ORDER])
print("Top option fits, by listener:")
display(scored[scored["listener"].notna()].pivot_table(index="listener", columns="method", values="top_fits", aggfunc=pct)[METHOD_ORDER])

Top option fits, by speaker:


method,beam 8,beam 16,beam 32,sample 16
speaker,,,,
man,92% (11/12),92% (11/12),92% (11/12),42% (5/12)
woman,0% (0/12),0% (0/12),0% (0/12),0% (0/12)


Top option fits, by listener:


method,beam 8,beam 16,beam 32,sample 16
listener,,,,
man,0% (0/6),0% (0/6),0% (0/6),17% (1/6)
woman,17% (1/6),17% (1/6),17% (1/6),17% (1/6)


### 8d. Practice rows vs test rows

If the two halves are very different, the 15 practice rows may not represent the rest well.

In [17]:
fit_table(scored, "split")

method,beam 8,beam 16,beam 32,sample 16
split,,,,
practice,86% (12/14),86% (12/14),86% (12/14),100% (14/14)
test,78% (31/40),82% (33/40),92% (37/40),80% (32/40)


## 9. Examples (practice rows only)

Two kinds of example:
- **A.** sentences where 32 options found a fit that 8 options missed (asking for more helped),
- **B.** sentences where no method found a fit (more options didn't help).

For each we print the Mandarin, the labels, NLLB's top option (beam 8, #1) and the best fitting option if any, with any wrong-word warning.

Following the fair-test rule we only print **practice** rows here; for test rows we only give the count. Set `EXAMPLE_SPLIT = "all"` to see every row (only once the detection rules are final).

In [18]:
EXAMPLE_SPLIT = "practice"

def options_for(row_id, method):
    o = options_table[(options_table["id"] == row_id) & (options_table["method"] == method)]
    return o.sort_values("rank")["option"].tolist()

def show(row_id):
    row = sentences.set_index("id").loc[row_id]
    print(f"{row_id}  {row['mandarin']}   speaker {row['speaker_gender']}, listener {row['listener_gender']}, "
          f"formal {row['formal']}  ->  must show {row['need']}")
    top = options_for(row_id, "beam 8")[0]
    print(f"    top option : {top}   (detected: speaker {detect_speaker_gender(top)}, "
          f"listener {detect_listener_gender(top)}, formality {detect_formality(top)})")
    for method in ["beam 32", "beam 16", "sample 16", "beam 8"]:
        opts = options_for(row_id, method)
        fit = [(i, o) for i, o in enumerate(opts, 1) if fits(o, row["need"])]
        if fit:
            rank, option = fit[0]
            hints = wrong_word_hints(option)
            print(f"    best fit   : {option}   ({method}, #{rank} of {len(opts)})" + (f"   WARNING: {hints}" if hints else ""))
            return
    print("    best fit   : none of the methods produced a fitting option")

wide = scored.pivot_table(index="id", columns="method", values="some_fit", aggfunc="first")
split_of = sentences.set_index("id")["split"]
helped = [i for i in wide.index if wide.loc[i, "beam 32"] and not wide.loc[i, "beam 8"]]
none_fit = [i for i in wide.index if not wide.loc[i].any()]

for title, ids in [("A. 32 options found a fit that 8 missed", helped), ("B. No method found a fit", none_fit)]:
    shown = [i for i in ids if EXAMPLE_SPLIT == "all" or split_of[i] == EXAMPLE_SPLIT]
    print(f"\n=== {title}: {len(ids)} sentences in total "
          f"({len(shown)} {EXAMPLE_SPLIT}, {len(ids) - len(shown)} hidden) ===\n")
    hidden = [i for i in ids if i not in shown]
    if hidden:
        print("Hidden rows by what they test:", dict(Counter(sentences.set_index("id").loc[hidden, "what_it_tests"])))
    for i in shown[:5]:
        show(i)


=== A. 32 options found a fit that 8 missed: 7 sentences in total (0 practice, 7 hidden) ===

Hidden rows by what they test: {'speaker_gender': 3, 'listener_gender': 1, 'formality': 1, 'speaker_gender+formality': 2}

=== B. No method found a fit: 2 sentences in total (0 practice, 2 hidden) ===

Hidden rows by what they test: {'speaker_gender': 1, 'listener_gender+formality': 1}


### 9b. What the detectors read on every practice sentence

To catch detector mistakes, this lists each practice sentence's top option and the first fitting option from beam 32, with what the detectors read in them. A Serbian speaker should check this list: a wrong "fits" here means the detector is wrong, not the model.

In [19]:
for row_id in sentences.loc[(sentences["split"] == "practice") & (sentences["what_it_tests"] != "control"), "id"]:
    show(row_id)

t01  我昨天加了两个小时的班。   speaker woman, listener man, formal yes  ->  must show {'speaker': 'woman'}
    top option : U subotu sam dodao dva sata.   (detected: speaker man, listener None, formality None)
    best fit   : Ali, ja sam uradila nešto na svom nivou.   (sample 16, #5 of 16)
t02  我把报告写完了。   speaker man, listener woman, formal yes  ->  must show {'speaker': 'man'}
    top option : Spisao sam izveštaj.   (detected: speaker man, listener None, formality None)
    best fit   : Spisao sam izveštaj.   (beam 32, #1 of 24)
t03  我已经到车间了。   speaker woman, listener woman, formal no  ->  must show {'speaker': 'woman'}
    top option : Došao sam u radionicu.   (detected: speaker man, listener None, formality None)
    best fit   : Došla sam do radionice.   (beam 32, #2 of 32)
t06  我刚才看见叉车撞到货架了。   speaker man, listener woman, formal yes  ->  must show {'speaker': 'man'}
    top option : To što sam video je da se vinka udari u policu.   (detected: speaker man, listener None, formality None)
    

### 9c. Wrong-word warnings among fitting options

The checker from section 7 flags words that aren't Serbian (včera, učora, Russian-style "proveril", and Russian or Ukrainian letters like я or ы, which `to_latin()` leaves unchanged). It can't catch a wrong meaning in general (uključio = "turned on" instead of "turned off" was found by a person and added to its list). Below: how often the *first fitting* option of each method has a warning, over all rows, then the practice-row cases to show our Serbian speaker.

In [20]:
warned = []
for _, row in sentences[sentences["need"].astype(bool)].iterrows():
    for method in METHOD_ORDER:
        fit = [o for o in options_for(row["id"], method) if fits(o, row["need"])]
        if fit:
            warned.append({"id": row["id"], "split": row["split"], "method": method,
                           "first fitting option": fit[0], "warnings": wrong_word_hints(fit[0])})
warned = pd.DataFrame(warned)
warned["has warning"] = warned["warnings"].str.len() > 0
print(warned.groupby("method", sort=False)["has warning"].apply(pct).rename("first fitting option has a warning").to_string())
practice_warned = warned[warned["has warning"] & (warned["split"] == "practice")]
practice_warned[["id", "method", "first fitting option", "warnings"]]

method
sample 16    0% (0/46)
beam 8       0% (0/43)
beam 16      2% (1/45)
beam 32      2% (1/49)


,id,method,first fitting option,warnings


## 10. What we learned (first run, Sat Nov 14)

Numbers from the run saved in this notebook (CPU). Rough detectors, AI-drafted sentences, no human-checked Serbian yet, so treat them as a first look, not a final score.

- **NLLB on its own never made a woman speaker sound like a woman.** The top option fit for 0 of 12 woman speakers and 11 of 12 man speakers. That is the bias the project is about.
- **More options help, mostly for women.** Some option fits: beam 8 80% (43/54), beam 16 83%, beam 32 91% (49/54), sample 16 85%. For woman speakers: 6 → 8 → 10 of 12 (beam 8 → 16 → 32). Man speakers were already 11-12 of 12.
- **Beam 32 is the best single method, and still cheap**: 0.9 s per sentence on CPU vs 0.6 s for beam 8. Sampling was slower (2.2 s) and its fits can drift far from the meaning (practice row t01's only "fit" is "Ali, ja sam uradila nešto na svom nivou", "But I did something at my level").
- **Asking for more options doesn't help NLLB's top choice.** The top option fits about half the sentences (26-27 of 54) whichever beam size we use. When it misses, the first fitting option sits at rank 3-5 (median).
- **It's not enough on its own.** 5 of 54 sentences still have no fitting option in 32; 2 have none with any method. Those need a repair rule (for example changing -o sam to -la sam) or must be flagged "had to guess".
- **"Fits" isn't "correct".** Fitting options in the practice rows include wrong meanings: "U subotu" (on Saturday) for 昨天 (yesterday), "ruke" (hands) for gloves, "vinka" (not a word) for forklift, "Koliko si otišla juče?" ("How much did you leave yesterday?"). Our wrong-word checker flags only foreign or made-up words it has on its list (it warned on just 1 first-fitting option out of 49 for beam 32) and none of these wrong meanings, so our Serbian speaker has to check meaning, not just endings.

## Credits

- **Model:** NLLB-200 distilled 600M by Meta AI, [facebook/nllb-200-distilled-600M](https://huggingface.co/facebook/nllb-200-distilled-600M), CC BY-NC 4.0.
- **Libraries:** Hugging Face transformers, PyTorch, pandas, NumPy, sentencepiece, sacremoses.
- **Data:** our own 60 draft Mandarin sentences (AI-drafted, labels by the team), `data/draft_mandarin_60.csv` in this repo.
- The Serbian-ending detectors and the Cyrillic-to-Latin converter here are our own rough first version.